In [41]:
import os 
import joblib 
import numpy as np 
import pandas as pd 
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn .pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor


In [42]:
MODEL_FILE = "model_gurgau.pkl"
PIPELINE_FILE = "pipeline_gurgau.pkl"# BEACUSE incoming data tho filter nahi hoga thos piopeline ko save ka liya 
def buildin_pipeline(num_attribute,cat_attribute):
    num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("std_scaler", StandardScaler())])
    cat_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder())])
    full_pipeline = ColumnTransformer([("num", num_pipeline, num_attribute), ("cat", cat_pipeline, cat_attribute)])
    return full_pipeline
if not os.path.exists(MODEL_FILE) or not os.path.exists(PIPELINE_FILE):
    housing = pd.read_csv("housing.csv")
    housing["income_cat"] = pd.cut(housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5])
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    for train_index, test_index in split.split(housing, housing["income_cat"]):
        strat_train_set = housing.loc[train_index]
        strat_test_set = housing.loc[test_index].to_csv("input_data.csv", index=False)
        housing = strat_train_set.copy()
       
    housing_labels = housing["median_house_value"].copy()
    housing = housing.drop("median_house_value", axis=1)
    housing = housing.drop("income_cat", axis=1)
    num_attribute = housing.select_dtypes(include=[np.number]).columns.tolist()
    cat_attribute = housing.select_dtypes(exclude=[np.number]).columns.tolist()
    full_pipeline = buildin_pipeline(num_attribute, cat_attribute)
    housing_pr = full_pipeline.fit_transform(housing)
    print(housing_pr)
    model = randome_forest = RandomForestRegressor(random_state=42)
    model.fit(housing_pr, housing_labels )
    joblib.dump(model, MODEL_FILE)
    joblib.dump(full_pipeline, PIPELINE_FILE)
    print("model is trained and saved succesfully   ")
else:
    model = joblib.load(MODEL_FILE)
    full_pipeline = joblib.load(PIPELINE_FILE)
    input_data= pd.read_csv("input_data.csv")
    transformed_input= full_pipeline.transform(input_data)
    predictions=model.predict(transformed_input)
    input_data["median_house_value"]=predictions
    input_data.to_csv("predictions.csv", index=False)
    print("model and pipeline are loaded successfully")

model and pipeline are loaded successfully


In [43]:
predictions_data = pd.read_csv("predictions.csv")
#add the predictions_data["median_house_value"] column to the input_data.csv file to compare the predictions with the actual values 

input_data = pd.read_csv("input_data.csv")
input_data["predicted_median_house_value"] = predictions_data["median_house_value"]

input_data.to_csv("final_predictions.csv", index=False)
